# 11 — Integração final: reunião → oportunidade → RAG fundamentado

Este notebook conecta o classificador BERTimbau aos documentos da base TOTVS. Ele produz um relatório estruturado por reunião com probabilidade, produtos/temas recuperados, evidências, fontes e políticas de uso.

**Privacidade:** nenhuma transcrição ou trecho é salvo nos relatórios. **Limite:** os rótulos de treinamento ainda são pseudo-rótulos; portanto o resultado é apoio à revisão comercial, não decisão automática.

In [1]:
from __future__ import annotations
import json, math, re, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
SEED=42; CHUNK_THRESHOLD=.80; MIN_SUPPORTING_CHUNKS=2; MIN_SUPPORT_DENSITY=.05; TOP_CHUNKS=3; TOP_DOCS=3; MAX_LENGTH=512
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); BATCH_SIZE=32 if DEVICE.type=='cuda' else 4
def find_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'data').exists() and (p/'notebooks').exists(): return p
    raise FileNotFoundError('Raiz do projeto não encontrada.')
ROOT=find_root(); CHUNKS_PATH=ROOT/'data/processed/chunks_bertimbau.jsonl'; MODEL_PATH=ROOT/'data/processed/bertimbau_opportunity_best'
KB_PATH=ROOT/'data/knowledge_base/totvs_rag_kb_v1.json'; ALIASES_PATH=ROOT/'data/knowledge_base/rag_aliases.json'
RAG09_PATH=ROOT/'reports/metrics/rag_retrieval_evaluation.json'; RAG10_PATH=ROOT/'reports/metrics/sentence_embeddings_retrieval.json'
E5_EMBEDDINGS_PATH=ROOT/'data/processed/rag_multilingual_e5_small_embeddings.npz'; E5_MODEL_NAME='intfloat/multilingual-e5-small'
OUTPUT_PATH=ROOT/'data/processed/meeting_commercial_insights.jsonl'; REPORT_PATH=ROOT/'reports/metrics/final_integration_summary.json'
print({'device':str(DEVICE),'chunks_file':CHUNKS_PATH.exists(),'classifier':MODEL_PATH.exists()})

C:\Users\Gabriel Pereira\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cuda', 'chunks_file': True, 'classifier': True}


## Classificação dos chunks
Todos os chunks são avaliados em lotes. Para reduzir falsos alarmes em reuniões longas, uma reunião candidata precisa de pelo menos dois chunks com probabilidade ≥ 0,80 e densidade mínima de 5%. Esses limiares são heurísticos até a auditoria humana. Guardamos somente índices, probabilidades e metadados não textuais.

In [2]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as f: return [json.loads(line) for line in f if line.strip()]
chunks=read_jsonl(CHUNKS_PATH); tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH); classifier=AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE).eval()
probabilities=[]; started=time.perf_counter()
for start in range(0,len(chunks),BATCH_SIZE):
    batch=[x['text'] for x in chunks[start:start+BATCH_SIZE]]
    encoded=tokenizer(batch,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'):
        logits=classifier(**encoded).logits
    probabilities.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().tolist())
assert len(probabilities)==len(chunks); classification_seconds=time.perf_counter()-started
by_meeting=defaultdict(list)
for chunk,prob in zip(chunks,probabilities): by_meeting[str(chunk['meeting_id'])].append((float(prob),chunk))
print({'chunks':len(chunks),'meetings':len(by_meeting),'seconds':round(classification_seconds,2)})

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  11%|█         | 22/201 [00:00<00:00, 200.00it/s]

Loading weights:  24%|██▍       | 48/201 [00:00<00:00, 232.59it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 755.64it/s]

{'chunks': 29972, 'meetings': 1126, 'seconds': 392.58}


## Recuperação e políticas de grounding
O notebook lê e usa o vencedor do experimento 10. O E5 especializado usa o índice vetorial persistido; o BM25 permanece como fallback explícito para ambientes sem suporte ao vencedor.

In [3]:
kb=json.loads(KB_PATH.read_text(encoding='utf-8')); aliases=json.loads(ALIASES_PATH.read_text(encoding='utf-8'))['groups']; doc_ids=[d['id'] for d in kb]
rag09=json.loads(RAG09_PATH.read_text(encoding='utf-8')); rag10=json.loads(RAG10_PATH.read_text(encoding='utf-8'))
experiment_winner=rag10['best_retriever']; implemented_retriever=experiment_winner; fallback_used=False
STOP={'a','ao','aos','as','com','como','da','das','de','do','dos','e','em','entre','essa','esse','esta','este','foi','isso','mais','mas','muito','na','não','nas','no','nos','o','os','ou','para','pela','pelo','por','qual','que','se','sem','ser','sua','são','tem','um','uma','você','vs'}
def norm(s):
    s=unicodedata.normalize('NFKD',str(s).casefold()); return ''.join(c for c in s if not unicodedata.combining(c))
alias_variants=[[norm(x) for x in [g['canonical'],*g['aliases']]] for g in aliases]
def tokens(text,expand=True):
    n=norm(text); out=[x for x in re.findall(r'[a-z0-9+]+',n) if len(x)>=2 and x not in STOP]
    if expand:
        padded=f' {n} '
        for variants in alias_variants:
            if any(f' {v} ' in padded for v in variants): out.extend(tokens(' '.join(variants),False))
    return out
def weighted(d):
    fields=[(d.get('title',''),3),(d.get('product',''),3),(' '.join(d.get('keywords',[])),2),(d.get('category',''),1),(' '.join(d.get('segments',[])),1),(' '.join(d.get('related_products',[])),1),(' '.join(d.get('competitors',[])),2),(d.get('content',''),1)]
    return [t for value,w in fields for t in tokens(value)*w]
doc_tokens=[weighted(d) for d in kb]; tf=[Counter(x) for x in doc_tokens]; lengths=np.array([len(x) for x in doc_tokens],float); avg=lengths.mean(); df=Counter()
for x in doc_tokens: df.update(set(x))
def bm25_scores(text):
    scores=np.zeros(len(kb))
    for term in tokens(text):
        n=df.get(term,0)
        if not n: continue
        idf=math.log(1+(len(kb)-n+.5)/(n+.5))
        for i,freqs in enumerate(tf):
            f=freqs.get(term,0)
            if f: scores[i]+=idf*f*2.5/(f+1.5*(.25+.75*lengths[i]/avg))
    return scores
e5_tokenizer=e5_model=e5_doc_embeddings=None
if experiment_winner=='multilingual_e5_small':
    saved=np.load(E5_EMBEDDINGS_PATH); assert saved['document_ids'].tolist()==doc_ids
    e5_doc_embeddings=saved['embeddings']; e5_tokenizer=AutoTokenizer.from_pretrained(E5_MODEL_NAME); e5_model=AutoModel.from_pretrained(E5_MODEL_NAME).to(DEVICE).eval()
elif experiment_winner not in {'bm25_aliases'}:
    implemented_retriever='bm25_aliases'; fallback_used=True
def retrieve_many(texts,top_k=TOP_DOCS):
    if implemented_retriever=='multilingual_e5_small':
        vectors=[]
        for start in range(0,len(texts),BATCH_SIZE):
            batch=['query: '+x for x in texts[start:start+BATCH_SIZE]]; encoded=e5_tokenizer(batch,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
            with torch.inference_mode(), torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=='cuda'): hidden=e5_model(**encoded).last_hidden_state
            mask=encoded['attention_mask'].unsqueeze(-1); pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1); vectors.append(F.normalize(pooled,p=2,dim=1).cpu().numpy())
        score_matrix=np.concatenate(vectors)@e5_doc_embeddings.T
        return [[kb[i] for i in np.argsort(-scores,kind='stable')[:top_k]] for scores in score_matrix]
    return [[kb[i] for i in np.argsort(-bm25_scores(text),kind='stable')[:top_k]] for text in texts]
def policy(d):
    level=norm(d.get('evidence_level','')); sources=d.get('sources',[])
    if not sources: return 'context_only_do_not_state_as_fact_without_source'
    if 'hipotese' in level: return 'state_only_as_hypothesis_with_citation'
    if 'recomendacao' in level or 'validacao' in level: return 'state_as_recommendation_or_pending_validation_with_citation'
    return 'factual_claim_requires_citation'
print({'experiment_winner':experiment_winner,'implemented_retriever':implemented_retriever,'fallback_used':fallback_used})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2802.87it/s]

{'experiment_winner': 'multilingual_e5_small', 'implemented_retriever': 'multilingual_e5_small', 'fallback_used': False}


## Agregação e insights por reunião
A decisão combina quantidade e densidade de chunks fortes; o máximo e a média dos três maiores scores ficam como indicadores auxiliares. Uma reunião candidata é marcada como contexto misto quando também possui ao menos dois chunks fortemente negativos. Documentos sem fonte podem ajudar na triagem, mas não geram afirmações factuais.

In [4]:
top_by_meeting={meeting_id:sorted(items,key=lambda x:x[0],reverse=True)[:TOP_CHUNKS] for meeting_id,items in by_meeting.items()}
def meeting_is_candidate(items):
    supporting=sum(prob>=CHUNK_THRESHOLD for prob,_ in items); return supporting>=MIN_SUPPORTING_CHUNKS and supporting/len(items)>=MIN_SUPPORT_DENSITY
retrieval_requests=[(meeting_id,chunk['chunk_index'],chunk['text']) for meeting_id,top in top_by_meeting.items() if meeting_is_candidate(by_meeting[meeting_id]) for _,chunk in top]
retrieval_batches=retrieve_many([x[2] for x in retrieval_requests]) if retrieval_requests else []
retrieval_lookup={(meeting_id,chunk_index):docs for (meeting_id,chunk_index,_),docs in zip(retrieval_requests,retrieval_batches)}
rows=[]
for meeting_id,items in by_meeting.items():
    top=top_by_meeting[meeting_id]; max_prob=top[0][0]; top_mean=float(np.mean([x[0] for x in top])); supporting=sum(prob>=CHUNK_THRESHOLD for prob,_ in items); density=supporting/len(items); positive=meeting_is_candidate(items)
    conflict=positive and sum(prob<=.20 for prob,_ in items)>=2
    retrieved=[]; seen=set()
    if positive:
        for prob,chunk in top:
            for d in retrieval_lookup[(meeting_id,chunk['chunk_index'])]:
                if d['id'] not in seen: retrieved.append(d); seen.add(d['id'])
    evidence=[]
    for d in retrieved[:5]:
        urls=[s['url'] for s in d.get('sources',[])]; p=policy(d)
        evidence.append({'document_id':d['id'],'title':d.get('title'),'product':d.get('product'),'category':d.get('category'),'evidence_level':d.get('evidence_level'),'source_urls':urls,'claim_policy':p,'can_support_fact':bool(urls) and p=='factual_claim_requires_citation'})
    factual=[e for e in evidence if e['can_support_fact']]; hypotheses=[e for e in evidence if e['claim_policy']=='state_only_as_hypothesis_with_citation']
    insight={'status':'oportunidade_para_revisao' if positive else 'sem_oportunidade_detectada','summary':('Possível oportunidade comercial detectada; revisar os produtos e evidências recuperados.' if positive else 'Nenhum chunk ultrapassou o limiar de oportunidade.'),'factual_documents':[e['document_id'] for e in factual],'hypothesis_documents':[e['document_id'] for e in hypotheses],'citation_required':bool(factual or hypotheses)}
    rows.append({'meeting_id':meeting_id,'opportunity_probability_max':round(max_prob,6),'opportunity_probability_top3_mean':round(top_mean,6),'high_opportunity_chunk_count':supporting,'high_opportunity_chunk_density':round(density,6),'opportunity_label':int(positive),'conflicting_chunks':conflict,'supporting_chunk_indices':[x[1]['chunk_index'] for x in top] if positive else [],'retriever':implemented_retriever,'retrieval_evidence':evidence,'commercial_insight':insight})
rows.sort(key=lambda x:(-x['opportunity_probability_max'],x['meeting_id']))
OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True)
with OUTPUT_PATH.open('w',encoding='utf-8') as f:
    for row in rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
print({'meetings_written':len(rows),'opportunities':sum(x['opportunity_label'] for x in rows),'conflicts':sum(x['conflicting_chunks'] for x in rows)})

{'meetings_written': 1126, 'opportunities': 975, 'conflicts': 935}


## Validações do contrato e relatório
Estas validações medem integridade, privacidade e grounding — não precisão de negócio. A avaliação ponta a ponta continuará pendente até existirem reuniões anotadas por humanos.

In [5]:
assert len(rows)==len(by_meeting)==len({x['meeting_id'] for x in rows})
assert all('text' not in row and 'transcription' not in row for row in rows)
assert all(e['source_urls'] or not e['can_support_fact'] for row in rows for e in row['retrieval_evidence'])
assert all(not (e['claim_policy']=='state_only_as_hypothesis_with_citation' and e['can_support_fact']) for row in rows for e in row['retrieval_evidence'])
opportunities=[x for x in rows if x['opportunity_label']==1]; grounded=[x for x in opportunities if any(e['source_urls'] for e in x['retrieval_evidence'])]
summary={'schema_version':'1.0','pipeline':f'BERTimbau opportunity classifier -> meeting aggregation -> {implemented_retriever} -> grounded structured insight','chunks':len(chunks),'meetings':len(rows),'opportunity_meetings':len(opportunities),'opportunity_rate':len(opportunities)/len(rows),'conflicting_meetings':sum(x['conflicting_chunks'] for x in rows),'grounded_opportunity_meetings':len(grounded),'grounded_coverage':len(grounded)/len(opportunities) if opportunities else 0,'thresholds':{'chunk_probability':CHUNK_THRESHOLD,'minimum_supporting_chunks':MIN_SUPPORTING_CHUNKS,'minimum_support_density':MIN_SUPPORT_DENSITY},'aggregation':{'decision':'minimum count and density of high-probability chunks','support':'maximum and mean of top 3 chunk probabilities','conflict':'candidate meeting with at least two chunks <= 0.20'},'retrieval':{'experiment_winner':experiment_winner,'implemented':implemented_retriever,'fallback_used':fallback_used},'contracts':{'transcript_text_not_saved':True,'sources_required_for_factual_claims':True,'hypotheses_never_marked_as_facts':True},'classification_seconds':classification_seconds,'warning':'Thresholds are operational heuristics. End-to-end metrics and production selection require human meeting labels.'}
REPORT_PATH.parent.mkdir(parents=True,exist_ok=True); REPORT_PATH.write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "schema_version": "1.0",
  "pipeline": "BERTimbau opportunity classifier -> meeting aggregation -> multilingual_e5_small -> grounded structured insight",
  "chunks": 29972,
  "meetings": 1126,
  "opportunity_meetings": 975,
  "opportunity_rate": 0.8658969804618117,
  "conflicting_meetings": 935,
  "grounded_opportunity_meetings": 975,
  "grounded_coverage": 1.0,
  "thresholds": {
    "chunk_probability": 0.8,
    "minimum_supporting_chunks": 2,
    "minimum_support_density": 0.05
  },
  "aggregation": {
    "decision": "minimum count and density of high-probability chunks",
    "support": "maximum and mean of top 3 chunk probabilities",
    "conflict": "candidate meeting with at least two chunks <= 0.20"
  },
  "retrieval": {
    "experiment_winner": "multilingual_e5_small",
    "implemented": "multilingual_e5_small",
    "fallback_used": false
  },
  "contracts": {
    "transcript_text_not_saved": true,
    "sources_required_for_factual_claims": true,
    "hypotheses_never_marked_

## Próximos passos obrigatórios
1. Anotar a fila humana e criar teste final por reunião. 2. Revisar as 32 consultas do RAG com especialista e ampliar o conjunto. 3. Avaliar qualidade dos insights, correção das fontes e taxa de abstenção. 4. Somente então decidir limiar, agregação e modelo para produção.